# PISA 2022 & HSLS:09 Interactive Visualizations (Redesigned)

This notebook generates 9 interactive Vega-Lite visualizations with meaningful driver-driven relationships:
- **Viz 1-3: PISA-only charts** (international education context)
- **Viz 4-6: HSLS-only charts** (US longitudinal data)
- **Viz 7-9: Combined charts** (US vs international comparison)

## Design Principle
Each visualization follows the pattern:
- **Driver (Left):** Categorical breakdown that allows selection
- **Driven (Right):** Shows a DIFFERENT variable/relationship that reveals new insights

In [ ]:
import pandas as pd
import altair as alt
from pathlib import Path
import json
import numpy as np

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../assets/json")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pisa_df = pd.read_csv(DATA_DIR / "pisa_subset.csv", low_memory=False)
hsls_df = pd.read_csv(DATA_DIR / "hsls_subset.csv", low_memory=False)

print(f"PISA: {len(pisa_df)} rows, {len(pisa_df.columns)} columns")
print(f"HSLS: {len(hsls_df)} rows, {len(hsls_df.columns)} columns")

In [ ]:
DARK_CONFIG = {
    "background": "#030712",
    "view": {"stroke": "transparent"},
    "axis": {
        "labelColor": "#E0E0E0",
        "titleColor": "#FFFFFF",
        "gridColor": "#333333",
        "domainColor": "#444444",
        "tickColor": "#444444"
    },
    "legend": {
        "labelColor": "#E0E0E0",
        "titleColor": "#FFFFFF"
    },
    "title": {"color": "#FFFFFF"}
}

OECD_COUNTRIES = ["AUS", "AUT", "BEL", "CAN", "CHE", "CHL", "COL", "CRI", "CZE", "DEU",
                  "DNK", "ESP", "EST", "FIN", "FRA", "GBR", "GRC", "HUN", "IRL", "ISL",
                  "ISR", "ITA", "JPN", "KOR", "LTU", "LUX", "LVA", "MEX", "NLD", "NOR",
                  "NZL", "POL", "PRT", "SVK", "SVN", "SWE", "TUR", "USA"]

def save_chart(chart, filename):
    spec = json.loads(chart.to_json())
    spec["config"] = DARK_CONFIG
    with open(OUTPUT_DIR / filename, "w") as f:
        json.dump(spec, f, indent=2)
    print(f"Saved: {filename}")

---
## PISA-ONLY VISUALIZATIONS (Viz 1-3)
---

### Viz 1: Does the Gender Confidence Gap Predict Achievement Gap?

**Left (Driver):** Countries ranked by gender gap in math CONFIDENCE (efficacy)  
**Right (Driven):** Gender gap in actual math ACHIEVEMENT for selected country

**Insight:** Reveals whether confidence gaps translate to achievement gaps

In [ ]:
pisa_gender = pisa_df[(pisa_df["MATHEFF"].notna()) & 
                       (pisa_df["PV1MATH"].notna()) & 
                       (pisa_df["ST004D01T"].isin([1, 2]))].copy()
pisa_gender["Gender"] = pisa_gender["ST004D01T"].map({1: "Female", 2: "Male"})

gender_efficacy = pisa_gender.groupby(["CNT", "Gender"]).agg({
    "MATHEFF": "mean",
    "PV1MATH": "mean"
}).reset_index()
gender_efficacy.columns = ["Country", "Gender", "Math_Efficacy", "Math_Score"]

gender_pivot = gender_efficacy.pivot(index="Country", columns="Gender", values="Math_Efficacy").reset_index()
gender_pivot["Efficacy_Gap"] = gender_pivot["Male"] - gender_pivot["Female"]
top_20 = gender_pivot.nlargest(20, "Efficacy_Gap")

efficacy_long = gender_efficacy[gender_efficacy["Country"].isin(top_20["Country"])].copy()
efficacy_long = efficacy_long.merge(top_20[["Country", "Efficacy_Gap"]], on="Country")

country_select = alt.selection_point(fields=["Country"], name="country_select")

left_chart = alt.Chart(efficacy_long).mark_bar(cornerRadiusTopRight=4, cornerRadiusBottomRight=4, cursor="pointer").encode(
    x=alt.X("Math_Efficacy:Q", title="Math Self-Efficacy (Confidence)", scale=alt.Scale(domain=[-1.0, 0.6])),
    y=alt.Y("Country:N", sort=alt.EncodingSortField(field="Efficacy_Gap", order="descending"), title=None),
    color=alt.Color("Gender:N", scale=alt.Scale(domain=["Female", "Male"], range=["#E91E63", "#1976D2"]),
                    legend=alt.Legend(title="Gender", orient="top")),
    xOffset="Gender:N",
    opacity=alt.condition(country_select, alt.value(1), alt.value(0.3)),
    tooltip=["Country:N", "Gender:N", alt.Tooltip("Math_Efficacy:Q", format=".2f", title="Confidence")]
).add_params(country_select).properties(
    name="view_1",
    title={"text": "Click Country: Confidence Gap", "color": "#FFFFFF", "fontSize": 14},
    width=450, height=350
)

right_chart = alt.Chart(efficacy_long).mark_bar(cornerRadiusTopRight=4, cornerRadiusBottomRight=4).encode(
    x=alt.X("Math_Score:Q", title="Actual Math Score (Achievement)", scale=alt.Scale(domain=[350, 600])),
    y=alt.Y("Gender:N", title=None),
    color=alt.Color("Gender:N", scale=alt.Scale(domain=["Female", "Male"], range=["#E91E63", "#1976D2"]),
                    legend=alt.Legend(title="Gender", orient="top")),
    tooltip=["Country:N", "Gender:N", alt.Tooltip("Math_Score:Q", format=".0f", title="Achievement")]
).transform_filter(country_select).properties(
    title={"text": "Achievement Gap for Selected Country", "color": "#FFFFFF", "fontSize": 14},
    width=450, height=350
)

viz1 = alt.hconcat(left_chart, right_chart).resolve_scale(color="independent")
save_chart(viz1, "pisa_gender_efficacy_dumbbell.json")
viz1

### Viz 2: How Does Math Anxiety Affect Performance Distribution?

**Left (Driver):** Student counts by anxiety level  
**Right (Driven):** Box plot showing math score DISTRIBUTION by gender

**Insight:** Shows variability in performance within each anxiety level

In [ ]:
pisa_anxiety = pisa_df[(pisa_df["ANXMAT"].notna()) & 
                        (pisa_df["PV1MATH"].notna()) &
                        (pisa_df["ST004D01T"].isin([1, 2]))].copy()
pisa_anxiety["Gender"] = pisa_anxiety["ST004D01T"].map({1: "Female", 2: "Male"})
pisa_anxiety["Anxiety_Level"] = pd.qcut(pisa_anxiety["ANXMAT"], 5, labels=["Very Low", "Low", "Medium", "High", "Very High"])

anxiety_counts = pisa_anxiety.groupby("Anxiety_Level").size().reset_index(name="Count")
anxiety_counts["Anxiety_Level"] = anxiety_counts["Anxiety_Level"].astype(str)

box_data = pisa_anxiety[["Anxiety_Level", "Gender", "PV1MATH"]].copy()
box_data.columns = ["Anxiety_Level", "Gender", "Math_Score"]
box_data["Anxiety_Level"] = box_data["Anxiety_Level"].astype(str)
box_sample = box_data.sample(n=min(4000, len(box_data)), random_state=42)

anxiety_order = ["Very Low", "Low", "Medium", "High", "Very High"]
anxiety_colors = ["#4575b4", "#91bfdb", "#fee090", "#fc8d59", "#d73027"]

anxiety_select = alt.selection_point(fields=["Anxiety_Level"], name="anxiety_select")

left_chart = alt.Chart(anxiety_counts).mark_bar(cornerRadiusTopRight=4, cornerRadiusBottomRight=4, cursor="pointer").encode(
    x=alt.X("Count:Q", title="Number of Students"),
    y=alt.Y("Anxiety_Level:N", title=None, sort=anxiety_order),
    color=alt.Color("Anxiety_Level:N", scale=alt.Scale(domain=anxiety_order, range=anxiety_colors),
                    legend=alt.Legend(title="Anxiety Level", orient="top")),
    opacity=alt.condition(anxiety_select, alt.value(1), alt.value(0.3)),
    tooltip=["Anxiety_Level:N", alt.Tooltip("Count:Q", format=",d")]
).add_params(anxiety_select).properties(
    name="view_1",
    title={"text": "Click Anxiety Level", "color": "#FFFFFF", "fontSize": 14},
    width=450, height=350
)

right_chart = alt.Chart(box_sample).mark_boxplot(extent="min-max", size=40).encode(
    x=alt.X("Math_Score:Q", title="Math Score Distribution", scale=alt.Scale(domain=[200, 800])),
    y=alt.Y("Gender:N", title=None),
    color=alt.Color("Gender:N", scale=alt.Scale(domain=["Female", "Male"], range=["#E91E63", "#1976D2"]),
                    legend=alt.Legend(title="Gender", orient="top"))
).transform_filter(anxiety_select).properties(
    title={"text": "Score Distribution by Gender", "color": "#FFFFFF", "fontSize": 14},
    width=450, height=350
)

viz2 = alt.hconcat(left_chart, right_chart).resolve_scale(color="independent")
save_chart(viz2, "pisa_anxiety_performance_heatmap.json")
viz2

### Viz 3: Immigration Status: Does Belonging Predict Achievement?

**Left (Driver):** Average sense of belonging by immigration status  
**Right (Driven):** Scatter plot showing BELONGING vs ACHIEVEMENT relationship

**Insight:** Reveals if feeling included matters more for certain groups

In [ ]:
pisa_immig = pisa_df[(pisa_df["IMMIG"].isin([1, 2, 3])) & 
                      (pisa_df["PV1MATH"].notna()) &
                      (pisa_df["BELONG"].notna()) &
                      (pisa_df["ST004D01T"].isin([1, 2]))].copy()
pisa_immig["Immigration_Status"] = pisa_immig["IMMIG"].map({1: "Native", 2: "Second-gen", 3: "First-gen"})
pisa_immig["Gender"] = pisa_immig["ST004D01T"].map({1: "Female", 2: "Male"})

immig_agg = pisa_immig.groupby("Immigration_Status").agg({
    "BELONG": "mean",
    "PV1MATH": ["mean", "count"]
}).reset_index()
immig_agg.columns = ["Immigration_Status", "Avg_Belonging", "Avg_Math", "Count"]

scatter_data = pisa_immig[["Immigration_Status", "Gender", "BELONG", "PV1MATH"]].copy()
scatter_data.columns = ["Immigration_Status", "Gender", "Belonging", "Math_Score"]
scatter_sample = scatter_data.sample(n=min(5000, len(scatter_data)), random_state=42)

immig_order = ["Native", "Second-gen", "First-gen"]
immig_colors = ["#0072B2", "#56B4E9", "#D55E00"]

immig_select = alt.selection_point(fields=["Immigration_Status"], name="immig_select")

left_chart = alt.Chart(immig_agg).mark_bar(cornerRadiusTopRight=4, cornerRadiusBottomRight=4, cursor="pointer").encode(
    x=alt.X("Avg_Belonging:Q", title="Average Sense of Belonging", scale=alt.Scale(domain=[-0.3, 0.3])),
    y=alt.Y("Immigration_Status:N", title=None, sort=immig_order),
    color=alt.Color("Immigration_Status:N", scale=alt.Scale(domain=immig_order, range=immig_colors),
                    legend=alt.Legend(title="Immigration Status", orient="top")),
    opacity=alt.condition(immig_select, alt.value(1), alt.value(0.3)),
    tooltip=["Immigration_Status:N", alt.Tooltip("Avg_Belonging:Q", format=".2f"), alt.Tooltip("Count:Q", format=",d")]
).add_params(immig_select).properties(
    name="view_1",
    title={"text": "Click Immigration Status", "color": "#FFFFFF", "fontSize": 14},
    width=450, height=350
)

right_chart = alt.Chart(scatter_sample).mark_circle(size=30, opacity=0.5).encode(
    x=alt.X("Belonging:Q", title="Sense of Belonging", scale=alt.Scale(domain=[-3, 3])),
    y=alt.Y("Math_Score:Q", title="Math Achievement", scale=alt.Scale(domain=[200, 800])),
    color=alt.Color("Gender:N", scale=alt.Scale(domain=["Female", "Male"], range=["#E91E63", "#1976D2"]),
                    legend=alt.Legend(title="Gender", orient="top")),
    tooltip=["Immigration_Status:N", "Gender:N", alt.Tooltip("Belonging:Q", format=".2f"), alt.Tooltip("Math_Score:Q", format=".0f")]
).transform_filter(immig_select).properties(
    title={"text": "Belonging vs Achievement", "color": "#FFFFFF", "fontSize": 14},
    width=450, height=350
)

viz3 = alt.hconcat(left_chart, right_chart).resolve_scale(color="independent")
save_chart(viz3, "combined_immigration.json")
viz3

---
## HSLS-ONLY VISUALIZATIONS (Viz 4-6)
---

### Viz 4: Does Parent Education Reduce Gender Gaps in STEM Aspirations?

**Left (Driver):** STEM career expectation rate by parent education level  
**Right (Driven):** Gender breakdown of STEM expectations for selected level

**Insight:** Tests if higher parental education reduces gender disparities

In [ ]:
hsls_stem = hsls_df[(hsls_df["X1PAR1EDU"].notna()) & 
                     (hsls_df["X1SEX"].notna()) & 
                     (hsls_df["X1STU30OCC_STEM1"].notna())].copy()
hsls_stem = hsls_stem[(hsls_stem["X1PAR1EDU"] > 0) & (hsls_stem["X1SEX"].isin([1, 2]))]

edu_map = {1: "Less than HS", 2: "HS Diploma", 3: "Some College", 4: "Associate's", 5: "Bachelor's", 6: "Master's", 7: "Ph.D./Prof"}
hsls_stem["Parent_Education"] = hsls_stem["X1PAR1EDU"].map(edu_map)
hsls_stem["Gender"] = hsls_stem["X1SEX"].map({1: "Male", 2: "Female"})

edu_agg = hsls_stem.groupby("Parent_Education").agg({"X1STU30OCC_STEM1": "mean", "X1SEX": "count"}).reset_index()
edu_agg.columns = ["Parent_Education", "STEM_Rate", "Count"]
edu_agg["STEM_Rate"] = edu_agg["STEM_Rate"] * 100

gender_edu_agg = hsls_stem.groupby(["Parent_Education", "Gender"]).agg({"X1STU30OCC_STEM1": "mean", "X1SEX": "count"}).reset_index()
gender_edu_agg.columns = ["Parent_Education", "Gender", "STEM_Rate", "Count"]
gender_edu_agg["STEM_Rate"] = gender_edu_agg["STEM_Rate"] * 100

edu_order = ["Less than HS", "HS Diploma", "Some College", "Associate's", "Bachelor's", "Master's", "Ph.D./Prof"]
edu_colors = ["#e1bee7", "#ce93d8", "#ba68c8", "#ab47bc", "#9c27b0", "#7b1fa2", "#6a1b9a"]

edu_select = alt.selection_point(fields=["Parent_Education"], name="edu_select")

left_chart = alt.Chart(edu_agg).mark_bar(cornerRadiusTopRight=4, cornerRadiusBottomRight=4, cursor="pointer").encode(
    x=alt.X("STEM_Rate:Q", title="Overall STEM Expectation Rate (%)", scale=alt.Scale(domain=[0, 50])),
    y=alt.Y("Parent_Education:N", title=None, sort=edu_order),
    color=alt.Color("Parent_Education:N", scale=alt.Scale(domain=edu_order, range=edu_colors),
                    legend=alt.Legend(title="Parent Education", orient="top", columns=4)),
    opacity=alt.condition(edu_select, alt.value(1), alt.value(0.3)),
    tooltip=["Parent_Education:N", alt.Tooltip("STEM_Rate:Q", format=".1f"), alt.Tooltip("Count:Q", format=",d")]
).add_params(edu_select).properties(
    name="view_1",
    title={"text": "Click Education Level", "color": "#FFFFFF", "fontSize": 14},
    width=450, height=350
)

right_chart = alt.Chart(gender_edu_agg).mark_bar(cornerRadiusTopRight=4, cornerRadiusBottomRight=4).encode(
    x=alt.X("STEM_Rate:Q", title="STEM Career Expectation Rate (%)", scale=alt.Scale(domain=[0, 50])),
    y=alt.Y("Gender:N", title=None),
    color=alt.Color("Gender:N", scale=alt.Scale(domain=["Female", "Male"], range=["#E91E63", "#1976D2"]),
                    legend=alt.Legend(title="Gender", orient="top")),
    tooltip=["Parent_Education:N", "Gender:N", alt.Tooltip("STEM_Rate:Q", format=".1f"), alt.Tooltip("Count:Q", format=",d")]
).transform_filter(edu_select).properties(
    title={"text": "Gender Gap for Selected Level", "color": "#FFFFFF", "fontSize": 14},
    width=450, height=350
)

viz4 = alt.hconcat(left_chart, right_chart).resolve_scale(color="independent")
save_chart(viz4, "combined_gender_stem.json")
viz4

### Viz 5: Does Math Identity Predict Achievement Equally Across Groups?

**Left (Driver):** Average math identity by race/ethnicity  
**Right (Driven):** Scatter plot showing identity-to-achievement CORRELATION

**Insight:** Reveals if identity-building interventions work differently for different groups

In [ ]:
hsls_identity = hsls_df[(hsls_df["X1MTHID"].notna()) & 
                         (hsls_df["X1TXMTSCOR"].notna()) & 
                         (hsls_df["X1RACE"].notna()) &
                         (hsls_df["X1SEX"].isin([1, 2]))].copy()
hsls_identity = hsls_identity[(hsls_identity["X1RACE"] > 0) & 
                               (hsls_identity["X1MTHID"] >= -3) & 
                               (hsls_identity["X1MTHID"] <= 3) &
                               (hsls_identity["X1TXMTSCOR"] >= 0)]

race_map = {
    1: "Am. Indian/Alaska Native", 2: "Asian", 3: "Black/African American",
    4: "Hispanic", 5: "More than one race", 6: "Native Hawaiian/Pac. Isl.", 7: "White"
}
hsls_identity["Race"] = hsls_identity["X1RACE"].map(race_map)
hsls_identity["Gender"] = hsls_identity["X1SEX"].map({1: "Male", 2: "Female"})

race_agg = hsls_identity.groupby("Race").agg({"X1MTHID": "mean", "X1TXMTSCOR": "count"}).reset_index()
race_agg.columns = ["Race", "Math_Identity", "Count"]

scatter_data = hsls_identity[["Race", "Gender", "X1MTHID", "X1TXMTSCOR"]].copy()
scatter_data.columns = ["Race", "Gender", "Math_Identity", "Math_Score"]
scatter_sample = scatter_data.sample(n=min(5000, len(scatter_data)), random_state=42)

race_colors = ["#E69F00", "#56B4E9", "#009E73", "#F0E442", "#0072B2", "#D55E00", "#CC79A7"]
race_order = list(race_map.values())

race_select = alt.selection_point(fields=["Race"], name="race_select")

left_chart = alt.Chart(race_agg).mark_bar(cornerRadiusTopRight=4, cornerRadiusBottomRight=4, cursor="pointer").encode(
    x=alt.X("Math_Identity:Q", title="Average Math Identity", scale=alt.Scale(domain=[-0.5, 0.5])),
    y=alt.Y("Race:N", title=None, sort=race_order),
    color=alt.Color("Race:N", scale=alt.Scale(domain=race_order, range=race_colors), 
                    legend=alt.Legend(title="Race/Ethnicity", orient="top", columns=3)),
    opacity=alt.condition(race_select, alt.value(1), alt.value(0.3)),
    tooltip=["Race:N", alt.Tooltip("Math_Identity:Q", format=".2f"), alt.Tooltip("Count:Q", format=",d")]
).add_params(race_select).properties(
    name="view_1",
    title={"text": "Click Race/Ethnicity", "color": "#FFFFFF", "fontSize": 14},
    width=450, height=350
)

right_chart = alt.Chart(scatter_sample).mark_circle(size=30, opacity=0.5).encode(
    x=alt.X("Math_Identity:Q", title="Math Identity Score", scale=alt.Scale(domain=[-3, 3])),
    y=alt.Y("Math_Score:Q", title="Math Test Score", scale=alt.Scale(domain=[20, 80])),
    color=alt.Color("Gender:N", scale=alt.Scale(domain=["Female", "Male"], range=["#E91E63", "#1976D2"]),
                    legend=alt.Legend(title="Gender", orient="top")),
    tooltip=["Race:N", "Gender:N", alt.Tooltip("Math_Identity:Q", format=".2f"), alt.Tooltip("Math_Score:Q", format=".1f")]
).transform_filter(race_select).properties(
    title={"text": "Identity vs Achievement", "color": "#FFFFFF", "fontSize": 14},
    width=450, height=350
)

viz5 = alt.hconcat(left_chart, right_chart).resolve_scale(color="independent")
save_chart(viz5, "hsls_math_identity_race.json")
viz5

### Viz 6: Does SES Predict STEM Major Choice?

**Left (Driver):** 12th grade GPA by SES quintile  
**Right (Driven):** STEM major enrollment rate for selected SES level

**Insight:** Shows whether GPA differences translate to STEM outcomes

In [ ]:
hsls_ses = hsls_df[(hsls_df["X1SESQ5"].notna()) & 
                    (hsls_df["X3TGPA12TH"].notna()) &
                    (hsls_df["X4RFDGMJSTEM"].notna()) &
                    (hsls_df["X1SEX"].isin([1, 2]))].copy()
hsls_ses = hsls_ses[(hsls_ses["X1SESQ5"] > 0) & (hsls_ses["X3TGPA12TH"] > 0)]

ses_labels = {1: "Lowest 20%", 2: "Second 20%", 3: "Middle 20%", 4: "Fourth 20%", 5: "Highest 20%"}
hsls_ses["SES_Quintile"] = hsls_ses["X1SESQ5"].map(ses_labels)
hsls_ses["Gender"] = hsls_ses["X1SEX"].map({1: "Male", 2: "Female"})

ses_gpa = hsls_ses.groupby("SES_Quintile").agg({"X3TGPA12TH": "mean", "X1SESQ5": "count"}).reset_index()
ses_gpa.columns = ["SES_Quintile", "GPA", "Count"]

ses_stem = hsls_ses.groupby(["SES_Quintile", "Gender"]).agg({"X4RFDGMJSTEM": "mean", "X1SEX": "count"}).reset_index()
ses_stem.columns = ["SES_Quintile", "Gender", "STEM_Rate", "Count"]
ses_stem["STEM_Rate"] = ses_stem["STEM_Rate"] * 100

ses_order = ["Lowest 20%", "Second 20%", "Middle 20%", "Fourth 20%", "Highest 20%"]
ses_colors = ["#1a237e", "#303f9f", "#3f51b5", "#7986cb", "#c5cae9"]

ses_select = alt.selection_point(fields=["SES_Quintile"], name="ses_select")

left_chart = alt.Chart(ses_gpa).mark_bar(cornerRadiusTopRight=4, cornerRadiusBottomRight=4, cursor="pointer").encode(
    x=alt.X("GPA:Q", title="Average 12th Grade GPA", scale=alt.Scale(domain=[2.0, 3.5])),
    y=alt.Y("SES_Quintile:N", title=None, sort=ses_order),
    color=alt.Color("SES_Quintile:N", scale=alt.Scale(domain=ses_order, range=ses_colors), 
                    legend=alt.Legend(title="SES Quintile", orient="top", columns=3)),
    opacity=alt.condition(ses_select, alt.value(1), alt.value(0.3)),
    tooltip=["SES_Quintile:N", alt.Tooltip("GPA:Q", format=".2f"), alt.Tooltip("Count:Q", format=",d")]
).add_params(ses_select).properties(
    name="view_1",
    title={"text": "Click SES Level", "color": "#FFFFFF", "fontSize": 14},
    width=450, height=350
)

right_chart = alt.Chart(ses_stem).mark_bar(cornerRadiusTopRight=4, cornerRadiusBottomRight=4).encode(
    x=alt.X("STEM_Rate:Q", title="STEM Major Enrollment Rate (%)", scale=alt.Scale(domain=[0, 40])),
    y=alt.Y("Gender:N", title=None),
    color=alt.Color("Gender:N", scale=alt.Scale(domain=["Female", "Male"], range=["#E91E63", "#1976D2"]),
                    legend=alt.Legend(title="Gender", orient="top")),
    tooltip=["SES_Quintile:N", "Gender:N", alt.Tooltip("STEM_Rate:Q", format=".1f"), alt.Tooltip("Count:Q", format=",d")]
).transform_filter(ses_select).properties(
    title={"text": "STEM Major Rate by Gender", "color": "#FFFFFF", "fontSize": 14},
    width=450, height=350
)

viz6 = alt.hconcat(left_chart, right_chart).resolve_scale(color="independent")
save_chart(viz6, "hsls_gpa_ses_trajectory.json")
viz6

---
## COMBINED VISUALIZATIONS (Viz 7-9)
---

### Viz 7: Do Confidence Gaps Match Achievement Gaps?

**Left (Driver):** Data sources with sample sizes  
**Right (Driven):** Side-by-side comparison of efficacy gap vs achievement gap

**Insight:** Directly compares whether confidence gaps align with achievement gaps

In [ ]:
hsls_eff = hsls_df[(hsls_df["X1MTHEFF"].notna()) & 
                    (hsls_df["X1TXMTSCOR"].notna()) &
                    (hsls_df["X1SEX"].isin([1, 2]))].copy()
hsls_eff["Source"] = "HSLS (US 2009)"
hsls_eff["Gender"] = hsls_eff["X1SEX"].map({1: "Male", 2: "Female"})
hsls_eff["Efficacy_Z"] = (hsls_eff["X1MTHEFF"] - hsls_eff["X1MTHEFF"].mean()) / hsls_eff["X1MTHEFF"].std()
hsls_eff["Math_Z"] = (hsls_eff["X1TXMTSCOR"] - hsls_eff["X1TXMTSCOR"].mean()) / hsls_eff["X1TXMTSCOR"].std()

pisa_eff = pisa_df[(pisa_df["MATHEFF"].notna()) & 
                    (pisa_df["PV1MATH"].notna()) &
                    (pisa_df["ST004D01T"].isin([1, 2]))].copy()
pisa_eff["Gender"] = pisa_eff["ST004D01T"].map({1: "Female", 2: "Male"})
pisa_eff["Efficacy_Z"] = (pisa_eff["MATHEFF"] - pisa_eff["MATHEFF"].mean()) / pisa_eff["MATHEFF"].std()
pisa_eff["Math_Z"] = (pisa_eff["PV1MATH"] - pisa_eff["PV1MATH"].mean()) / pisa_eff["PV1MATH"].std()

pisa_us = pisa_eff[pisa_eff["CNT"] == "USA"].copy()
pisa_us["Source"] = "PISA US (2022)"
pisa_intl = pisa_eff[pisa_eff["CNT"] != "USA"].copy()
pisa_intl["Source"] = "PISA Intl (2022)"

source_count = pd.DataFrame({
    "Source": ["HSLS (US 2009)", "PISA US (2022)", "PISA Intl (2022)"],
    "Count": [len(hsls_eff), len(pisa_us), len(pisa_intl)]
})

def calc_gaps(df, source):
    male = df[df["Gender"] == "Male"]
    female = df[df["Gender"] == "Female"]
    return pd.DataFrame({
        "Source": [source, source],
        "Gap_Type": ["Efficacy Gap", "Achievement Gap"],
        "Gap": [male["Efficacy_Z"].mean() - female["Efficacy_Z"].mean(),
                male["Math_Z"].mean() - female["Math_Z"].mean()]
    })

gaps = pd.concat([calc_gaps(hsls_eff, "HSLS (US 2009)"),
                  calc_gaps(pisa_us, "PISA US (2022)"),
                  calc_gaps(pisa_intl, "PISA Intl (2022)")])

source_order = ["HSLS (US 2009)", "PISA US (2022)", "PISA Intl (2022)"]
source_colors = ["#E69F00", "#56B4E9", "#009E73"]
gap_colors = ["#9c27b0", "#00bcd4"]

source_select = alt.selection_point(fields=["Source"], name="source_select")

left_chart = alt.Chart(source_count).mark_bar(cornerRadiusTopRight=4, cornerRadiusBottomRight=4, cursor="pointer").encode(
    x=alt.X("Count:Q", title="Number of Students", scale=alt.Scale(domain=[0, 700000])),
    y=alt.Y("Source:N", title=None, sort=source_order),
    color=alt.Color("Source:N", scale=alt.Scale(domain=source_order, range=source_colors), 
                    legend=alt.Legend(title="Data Source", orient="top")),
    opacity=alt.condition(source_select, alt.value(1), alt.value(0.3)),
    tooltip=["Source:N", alt.Tooltip("Count:Q", format=",d")]
).add_params(source_select).properties(
    name="view_1",
    title={"text": "Click Data Source", "color": "#FFFFFF", "fontSize": 14},
    width=450, height=350
)

right_chart = alt.Chart(gaps).mark_bar(cornerRadiusTopRight=4, cornerRadiusBottomRight=4).encode(
    x=alt.X("Gap:Q", title="Gender Gap (Male - Female, Std. Units)", scale=alt.Scale(domain=[-0.2, 0.5])),
    y=alt.Y("Gap_Type:N", title=None, sort=["Efficacy Gap", "Achievement Gap"]),
    color=alt.Color("Gap_Type:N", scale=alt.Scale(domain=["Efficacy Gap", "Achievement Gap"], range=gap_colors),
                    legend=alt.Legend(title="Gap Type", orient="top")),
    tooltip=["Source:N", "Gap_Type:N", alt.Tooltip("Gap:Q", format=".3f")]
).transform_filter(source_select).properties(
    title={"text": "Confidence vs Achievement Gap", "color": "#FFFFFF", "fontSize": 14},
    width=450, height=350
)

viz7 = alt.hconcat(left_chart, right_chart).resolve_scale(color="independent")
save_chart(viz7, "combined_efficacy_comparison.json")
viz7

### Viz 8: How Large is the SES Achievement Gap?

**Left (Driver):** Student counts by data source  
**Right (Driven):** Dumbbell chart showing lowest vs highest SES achievement

**Insight:** Visualizes the magnitude of SES inequality

In [ ]:
hsls_ses = hsls_df[(hsls_df["X1SESQ5"].notna()) & (hsls_df["X1TXMTSCOR"].notna())].copy()
hsls_ses = hsls_ses[hsls_ses["X1SESQ5"] > 0]
hsls_ses["SES_Quintile"] = hsls_ses["X1SESQ5"].astype(int)
hsls_ses["Math_Z"] = (hsls_ses["X1TXMTSCOR"] - hsls_ses["X1TXMTSCOR"].mean()) / hsls_ses["X1TXMTSCOR"].std()
hsls_ses["Source"] = "HSLS (US 2009)"

pisa_ses = pisa_df[(pisa_df["ESCS"].notna()) & (pisa_df["PV1MATH"].notna())].copy()
pisa_ses["SES_Quintile"] = pd.qcut(pisa_ses["ESCS"], 5, labels=[1, 2, 3, 4, 5]).astype(int)
pisa_ses["Math_Z"] = (pisa_ses["PV1MATH"] - pisa_ses["PV1MATH"].mean()) / pisa_ses["PV1MATH"].std()

pisa_us_ses = pisa_ses[pisa_ses["CNT"] == "USA"].copy()
pisa_us_ses["Source"] = "PISA US (2022)"
pisa_intl_ses = pisa_ses[pisa_ses["CNT"] != "USA"].copy()
pisa_intl_ses["Source"] = "PISA Intl (2022)"

combined = pd.concat([hsls_ses[["Source", "SES_Quintile", "Math_Z"]], 
                      pisa_us_ses[["Source", "SES_Quintile", "Math_Z"]], 
                      pisa_intl_ses[["Source", "SES_Quintile", "Math_Z"]]])

source_count = combined.groupby("Source").size().reset_index(name="Count")

extremes = combined[combined["SES_Quintile"].isin([1, 5])].copy()
extremes["SES_Level"] = extremes["SES_Quintile"].map({1: "Lowest 20%", 5: "Highest 20%"})
ses_extremes = extremes.groupby(["Source", "SES_Level"]).agg({"Math_Z": "mean"}).reset_index()
ses_extremes.columns = ["Source", "SES_Level", "Math_Score"]

source_order = ["HSLS (US 2009)", "PISA US (2022)", "PISA Intl (2022)"]
source_colors = ["#E69F00", "#56B4E9", "#009E73"]
ses_level_colors = ["#1a237e", "#c5cae9"]

source_select = alt.selection_point(fields=["Source"], name="source_select")

left_chart = alt.Chart(source_count).mark_bar(cornerRadiusTopRight=4, cornerRadiusBottomRight=4, cursor="pointer").encode(
    x=alt.X("Count:Q", title="Number of Students", scale=alt.Scale(domain=[0, 700000])),
    y=alt.Y("Source:N", title=None, sort=source_order),
    color=alt.Color("Source:N", scale=alt.Scale(domain=source_order, range=source_colors), 
                    legend=alt.Legend(title="Data Source", orient="top")),
    opacity=alt.condition(source_select, alt.value(1), alt.value(0.3)),
    tooltip=["Source:N", alt.Tooltip("Count:Q", format=",d")]
).add_params(source_select).properties(
    name="view_1",
    title={"text": "Click Data Source", "color": "#FFFFFF", "fontSize": 14},
    width=450, height=350
)

right_chart = alt.Chart(ses_extremes).mark_bar(cornerRadiusTopRight=4, cornerRadiusBottomRight=4).encode(
    x=alt.X("Math_Score:Q", title="Standardized Math Score", scale=alt.Scale(domain=[-1.0, 1.0])),
    y=alt.Y("SES_Level:N", title=None, sort=["Lowest 20%", "Highest 20%"]),
    color=alt.Color("SES_Level:N", scale=alt.Scale(domain=["Lowest 20%", "Highest 20%"], range=ses_level_colors),
                    legend=alt.Legend(title="SES Level", orient="top")),
    tooltip=["Source:N", "SES_Level:N", alt.Tooltip("Math_Score:Q", format=".3f")]
).transform_filter(source_select).properties(
    title={"text": "SES Achievement Gap", "color": "#FFFFFF", "fontSize": 14},
    width=450, height=350
)

viz8 = alt.hconcat(left_chart, right_chart).resolve_scale(color="independent")
save_chart(viz8, "combined_ses_achievement.json")
viz8

### Viz 9: Parent Education Premium: US vs International

**Left (Driver):** Math achievement by parent education level  
**Right (Driven):** Achievement by data source for selected education level

**Insight:** Shows if parent education pays off more in US vs internationally

In [ ]:
hsls_edu_map = {1: "Less than HS", 2: "HS Diploma", 3: "Some College", 4: "Some College", 5: "Bachelor's", 6: "Graduate+", 7: "Graduate+"}
pisa_edu_map = {1: "Less than HS", 2: "Less than HS", 3: "HS Diploma", 4: "Some College", 5: "Some College", 6: "Bachelor's", 7: "Graduate+", 8: "Graduate+"}

hsls_edu = hsls_df[(hsls_df["X1PAR1EDU"].notna()) & (hsls_df["X1TXMTSCOR"].notna())].copy()
hsls_edu = hsls_edu[hsls_edu["X1PAR1EDU"] > 0]
hsls_edu["Parent_Education"] = hsls_edu["X1PAR1EDU"].map(hsls_edu_map)
hsls_edu["Math_Z"] = (hsls_edu["X1TXMTSCOR"] - hsls_edu["X1TXMTSCOR"].mean()) / hsls_edu["X1TXMTSCOR"].std()
hsls_edu["Source"] = "HSLS (US 2009)"

pisa_edu = pisa_df[(pisa_df["HISCED"].notna()) & (pisa_df["PV1MATH"].notna())].copy()
pisa_edu = pisa_edu[pisa_edu["HISCED"] > 0]
pisa_edu["Parent_Education"] = pisa_edu["HISCED"].map(pisa_edu_map)
pisa_edu["Math_Z"] = (pisa_edu["PV1MATH"] - pisa_edu["PV1MATH"].mean()) / pisa_edu["PV1MATH"].std()

pisa_us_edu = pisa_edu[pisa_edu["CNT"] == "USA"].copy()
pisa_us_edu["Source"] = "PISA US (2022)"
pisa_intl_edu = pisa_edu[pisa_edu["CNT"] != "USA"].copy()
pisa_intl_edu["Source"] = "PISA Intl (2022)"

combined = pd.concat([hsls_edu[["Source", "Parent_Education", "Math_Z"]], 
                      pisa_us_edu[["Source", "Parent_Education", "Math_Z"]], 
                      pisa_intl_edu[["Source", "Parent_Education", "Math_Z"]]])

edu_agg = combined.groupby(["Parent_Education", "Source"]).agg({"Math_Z": "mean"}).reset_index()
edu_agg.columns = ["Parent_Education", "Source", "Math_Score"]

edu_order = ["Less than HS", "HS Diploma", "Some College", "Bachelor's", "Graduate+"]
source_order = ["HSLS (US 2009)", "PISA US (2022)", "PISA Intl (2022)"]
source_colors = ["#E69F00", "#56B4E9", "#009E73"]

edu_select = alt.selection_point(fields=["Parent_Education"], name="edu_select")

left_chart = alt.Chart(edu_agg).mark_bar(cornerRadiusTopRight=4, cornerRadiusBottomRight=4, cursor="pointer").encode(
    x=alt.X("Math_Score:Q", title="Standardized Math Score", scale=alt.Scale(domain=[-1.0, 1.0])),
    y=alt.Y("Parent_Education:N", title=None, sort=edu_order),
    color=alt.Color("Source:N", scale=alt.Scale(domain=source_order, range=source_colors),
                    legend=alt.Legend(title="Data Source", orient="top")),
    xOffset="Source:N",
    opacity=alt.condition(edu_select, alt.value(1), alt.value(0.3)),
    tooltip=["Parent_Education:N", "Source:N", alt.Tooltip("Math_Score:Q", format=".2f")]
).add_params(edu_select).properties(
    name="view_1",
    title={"text": "Click Education Level", "color": "#FFFFFF", "fontSize": 14},
    width=450, height=350
)

right_chart = alt.Chart(edu_agg).mark_bar(cornerRadiusTopRight=4, cornerRadiusBottomRight=4).encode(
    x=alt.X("Math_Score:Q", title="Standardized Math Score", scale=alt.Scale(domain=[-1.0, 1.0])),
    y=alt.Y("Source:N", title=None, sort=source_order),
    color=alt.Color("Source:N", scale=alt.Scale(domain=source_order, range=source_colors), 
                    legend=alt.Legend(title="Data Source", orient="top")),
    tooltip=["Parent_Education:N", "Source:N", alt.Tooltip("Math_Score:Q", format=".2f")]
).transform_filter(edu_select).properties(
    title={"text": "Source Comparison for Selected Level", "color": "#FFFFFF", "fontSize": 14},
    width=450, height=350
)

viz9 = alt.hconcat(left_chart, right_chart).resolve_scale(color="independent")
save_chart(viz9, "combined_parent_education.json")
viz9

---
## Summary

All 9 interactive visualizations have been redesigned with meaningful driver-driven relationships:

**PISA-only (Viz 1-3):**
1. `pisa_gender_efficacy_dumbbell.json` - Confidence Gap -> Achievement Gap
2. `pisa_anxiety_performance_heatmap.json` - Anxiety Level -> Score Distribution
3. `combined_immigration.json` - Immigration Status -> Belonging vs Achievement

**HSLS-only (Viz 4-6):**
4. `combined_gender_stem.json` - Parent Education -> STEM by Gender
5. `hsls_math_identity_race.json` - Race -> Identity vs Achievement Scatter
6. `hsls_gpa_ses_trajectory.json` - SES -> STEM Major Rate

**Combined (Viz 7-9):**
7. `combined_efficacy_comparison.json` - Source -> Efficacy vs Achievement Gap
8. `combined_ses_achievement.json` - Source -> SES Extremes Comparison
9. `combined_parent_education.json` - Parent Ed -> Source Comparison

In [ ]:
import os
json_files = [
    "pisa_gender_efficacy_dumbbell.json",
    "pisa_anxiety_performance_heatmap.json",
    "combined_immigration.json",
    "combined_gender_stem.json",
    "hsls_math_identity_race.json",
    "hsls_gpa_ses_trajectory.json",
    "combined_efficacy_comparison.json",
    "combined_ses_achievement.json",
    "combined_parent_education.json"
]

print("Generated JSON files:")
for f in json_files:
    path = OUTPUT_DIR / f
    if path.exists():
        size = os.path.getsize(path)
        print(f"  {f}: {size/1024:.1f} KB")
    else:
        print(f"  {f}: NOT FOUND")